## 02a Experiment 1a: Category Membership - Attributes Task

### 0 Setup

In [41]:
import os
from openai import OpenAI
import random
import pandas as pd
import time
import csv
import ast

### 1 Functions

In [42]:
# Prompt Model to answer Questionnaire

def collect_output(client, prompt):
    response = client.chat.completions.create(model="deepseek-chat",messages=[{"role": "user","content": prompt}],stream=False)
    return response


### A. Attributes Task

In [43]:
# Category
category = "Clothing"

In [44]:
# CATEGORIES & ATTRIBUTES #from rosch1975c
df = pd.read_csv('01_catmemexp_attributestask_items4attributes_2.csv')
items_df = df[df['category'] == f"{category}"]
items_df.head()

,model,category,frequency,items
18,deepseek-chat,Clothing,most_frequent,"['boot', 'jean', 'sock', 'hoodie', 'sandal', '..."
19,deepseek-chat,Clothing,average_frequency,"['purse', 'cloak', 'tuxedo', 'fanny pack', 'ba..."
20,deepseek-chat,Clothing,least_frequent,"['patching', 'lapis lazuli', 'lungi', 'jasper'..."
21,gemini-3-pro,Clothing,most_frequent,"['boot', 'jean', 'sock', 'hoodie', 'sandal', '..."
22,gemini-3-pro,Clothing,average_frequency,"['khaki', 'culotte', 'kaftan', 'onesie', 'slac..."


In [ ]:
# Collect Answers from Model

items_reps = 10 #30

# retrieve API      
client = OpenAI(api_key='', base_url="https://api.deepseek.com")

prompt_outputs = pd.DataFrame(columns=['model', 'prompt', 'prompt_id', 'runtime', 'category', 'frequency','item','output'])
model_output = pd.DataFrame()

for rep in range(items_reps): # max rep == 30

    # Extract items and frequencies for each category and model
    items_df = items_df[items_df['model'] == f"deepseek-chat"]
    items_list = items_df['items'].tolist()
    items_list = [ast.literal_eval(item) for item in items_list if isinstance(item, str)]
    items_list = [item for sublist in items_list for item in sublist]

    frequency_list_tmp = items_df['frequency'].tolist()
    frequency_list = [freq for freq in frequency_list_tmp for _ in range(6)]
    paired_items = list(zip(items_list, frequency_list))

    random.shuffle(paired_items) # shuffle items, changing order of category presentation at each repetition
    shuffled_items_list, shuffled_range_list = zip(*paired_items)
    shuffled_items_list = list(shuffled_items_list)
    shuffled_range_list = list(shuffled_range_list)
        
    for i in range(len(shuffled_items_list)):

        current_prompt = f"""
        Write all of the attributes of the object "{shuffled_items_list[i]}" that you can think of. Write the attributes or characteristics you think are characteristic of the object "{shuffled_items_list[i]}".
        Do not add any further details, only write the items that answer the instruction.
        Provide your answer in plain text as a comma-separated list of adjectives.
        """
            
        start_time = time.time() 
        output = collect_output(client, current_prompt)
        end_time = time.time() 
        runtime = end_time - start_time
        model_output = pd.DataFrame({
            'model': 'deepseek-chat',
            'prompt': [current_prompt],
            'prompt_id': [6],
            'runtime': [runtime],
            'category': [category],
            'frequency': [shuffled_range_list[i]],
            'item': [shuffled_items_list[i]],
            'output': [output]})
            
        prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)

prompt_outputs.head()

C:\Users\AS\AppData\Local\Temp\ipykernel_8876\2088710116.py:52: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  prompt_outputs = pd.concat([prompt_outputs, model_output], ignore_index=True)


,model,prompt,prompt_id,runtime,category,frequency,item,output
0,deepseek-chat,\n Write all of the attributes of the o...,6,134.980729,Clothing,most_frequent,sock,ChatCompletion(id='2d03613a-8599-481f-a614-87d...
1,deepseek-chat,\n Write all of the attributes of the o...,6,4.073982,Clothing,least_frequent,patching,ChatCompletion(id='5dbb9145-473c-46d6-94c5-786...
2,deepseek-chat,\n Write all of the attributes of the o...,6,136.269166,Clothing,least_frequent,jasper,ChatCompletion(id='61858125-213f-422d-be55-002...
3,deepseek-chat,\n Write all of the attributes of the o...,6,4.119591,Clothing,most_frequent,hoodie,ChatCompletion(id='34bb705e-c7f0-4393-a235-567...
4,deepseek-chat,\n Write all of the attributes of the o...,6,3.985172,Clothing,least_frequent,lungi,ChatCompletion(id='36a27759-cadb-43c4-8a06-4c3...


In [46]:
prompt_outputs['output'] = prompt_outputs['output'].apply(lambda x: x.choices[0].message.content if x.choices else None)

In [47]:
prompt_outputs

,model,prompt,prompt_id,runtime,category,frequency,item,output
0,deepseek-chat,\n Write all of the attributes of the o...,6,134.980729,Clothing,most_frequent,sock,"soft, flexible, warm, absorbent, stretchy, dur..."
1,deepseek-chat,\n Write all of the attributes of the o...,6,4.073982,Clothing,least_frequent,patching,"temporary, corrective, makeshift, improvised, ..."
2,deepseek-chat,\n Write all of the attributes of the o...,6,136.269166,Clothing,least_frequent,jasper,"hard, opaque, mineral, gemstone, chalcedony, m..."
3,deepseek-chat,\n Write all of the attributes of the o...,6,4.119591,Clothing,most_frequent,hoodie,"casual, comfortable, warm, soft, hooded, zippe..."
4,deepseek-chat,\n Write all of the attributes of the o...,6,3.985172,Clothing,least_frequent,lungi,"traditional, garment, cloth, wrap-around, wais..."
...,...,...,...,...,...,...,...,...
175,deepseek-chat,\n Write all of the attributes of the o...,6,6.401899,Clothing,least_frequent,lungi,"traditional, garment, wraparound, tubular, uns..."
176,deepseek-chat,\n Write all of the attributes of the o...,6,4.669797,Clothing,least_frequent,patching,"temporary, makeshift, expedient, incomplete, p..."
177,deepseek-chat,\n Write all of the attributes of the o...,6,140.334409,Clothing,most_frequent,sock,"soft, flexible, warm, absorbent, stretchy, dur..."
178,deepseek-chat,\n Write all of the attributes of the o...,6,3.470790,Clothing,least_frequent,never,"abstract, absolute, eternal, final, impossible..."


In [48]:
len(prompt_outputs)

180

In [49]:
# take away NaN Answers
print("Number of rows:", len(prompt_outputs),"\n")
print("Number of rows after NA filtering:", len(prompt_outputs.dropna()))

Number of rows: 180 

Number of rows after NA filtering: 180


In [50]:
prompt_outputs.to_csv(f'01_catmemexp_attributestask_allmodels_raw_2.csv',index=False, quoting=csv.QUOTE_ALL, encoding='utf-8', mode='a')